# 00 — Setup & Install

Run this notebook **once per Colab session** (after mounting Drive) to install
packages and confirm the environment is ready.

**Order of notebooks:**
1. `00_setup_and_install.ipynb` ← you are here
2. `01_download_maestro.ipynb` (run once)
3. `02_build_cache.ipynb` (run once, ~30 min)
4. `03_verify_pipeline.ipynb` (run every session)
5. `04_explore_data.ipynb` (optional exploration)

## 1. Check GPU

Colab Pro provides a T4 GPU. Confirm it is available before proceeding.

In [ ]:
!nvidia-smi

Fri Mar 27 20:13:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version : {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")

GPU available: Tesla T4
CUDA version : 12.8


## 2. Mount Google Drive

Google Drive is **permanent storage** — files survive Colab session restarts.
`/content/` (local Colab disk) is **wiped** every session.

Store MAESTRO, the NPZ cache, and checkpoints on Drive so you never need to
re-download or re-preprocess after the first run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


## 3. Install packages

Install all Python dependencies.  This cell takes ~2 minutes on first run.
Subsequent sessions are faster because Colab caches some wheels.

In [ ]:
!pip install -q torch torchaudio pretty_midi pandas tqdm matplotlib pyyaml ipywidgets

## 4. Clone / upload repo

**Option A (recommended): `git clone`**  
If the repo is on GitHub, clone it to `/content/piano_amt`.

**Option B: manual upload**  
Use the Colab file browser (left panel → 📁 icon → Upload) to upload the
`piano_amt/` folder, then update the path below.

In [ ]:

import os
from getpass import getpass

REPO_DIR = '/content/AMT_FYP'

TOKEN = getpass('GitHub token: ')

if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://{TOKEN}@github.com/Mobinmo83/AMT_FYP.git {REPO_DIR}")
    print(f"Cloned into {REPO_DIR} ✓")
else:
    print(f"Repo already exists at {REPO_DIR}")
    os.system(f"cd {REPO_DIR} && git pull")

GitHub token: ··········
Cloned into /content/AMT_FYP ✓


In [ ]:
import sys
REPO_DIR = '/content/AMT_FYP'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"sys.path[0] = {sys.path[0]}")

sys.path[0] = /content/AMT_FYP


## 5. Verify import

Confirms `src/constants.py` is importable and values are correct.

In [ ]:

import sys
sys.path.insert(0, '/content/AMT_FYP/piano_amt')

from src.constants import N_MELS, FRAMES_PER_SECOND, N_KEYS, SAMPLE_RATE, HOP_LENGTH

In [ ]:


print(f"N_MELS          = {N_MELS}")
print(f"SAMPLE_RATE     = {SAMPLE_RATE}")
print(f"HOP_LENGTH      = {HOP_LENGTH}")
print(f"FRAMES_PER_SEC  = {FRAMES_PER_SECOND}")
print(f"N_KEYS          = {N_KEYS}")

assert N_MELS == 229, f"N_MELS={N_MELS}, expected 229"
assert SAMPLE_RATE == 16000
assert HOP_LENGTH == 512
assert abs(FRAMES_PER_SECOND - 31.25) < 1e-6
assert N_KEYS == 88

print("\n✓ All constants verified — environment is ready!")

N_MELS          = 229
SAMPLE_RATE     = 16000
HOP_LENGTH      = 512
FRAMES_PER_SEC  = 31.25
N_KEYS          = 88

✓ All constants verified — environment is ready!


## Set paths

In [ ]:
import os
import glob

DRIVE_ROOT   = "/content/drive/MyDrive/piano_amt"
MAESTRO_DIR  = f"{DRIVE_ROOT}/maestro-v3.0.0"
ZIP_PATH     = f"{DRIVE_ROOT}/maestro-v3.0.0.zip"
CACHE_DIR    = f'{DRIVE_ROOT}/cache'
MAESTRO_ROOT = f'{DRIVE_ROOT}/maestro-v3.0.0'

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root : {DRIVE_ROOT}")
print(f"Dataset dir: {MAESTRO_DIR}")
print(f"ZIP path   : {ZIP_PATH}")
print(f"Cache dir   : {CACHE_DIR}")


Drive root : /content/drive/MyDrive/piano_amt
Dataset dir: /content/drive/MyDrive/piano_amt/maestro-v3.0.0
ZIP path   : /content/drive/MyDrive/piano_amt/maestro-v3.0.0.zip
Cache dir   : /content/drive/MyDrive/piano_amt/cache


## Check if already downloaded

In [ ]:

csv_exists = bool(glob.glob(f"{MAESTRO_DIR}/*.csv"))
if os.path.exists(MAESTRO_DIR) and csv_exists:
    print("✓ MAESTRO already downloaded and extracted — skip to 'Verify dataset' below.")
else:
    print("Dataset not yet downloaded. Continue with the cells below.")


✓ MAESTRO already downloaded and extracted — skip to 'Verify dataset' below.


## Download (skip if already done)

Downloads from the official Google Magenta storage bucket.  
**Expected time: 10–30 minutes** depending on Colab network speed.  
The `-c` flag resumes interrupted downloads.

In [ ]:

if not os.path.exists(ZIP_PATH):
    print("Downloading MAESTRO v3.0.0 (~16 GB)...")
    !wget -c "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0.zip" \
         -O "{ZIP_PATH}"
    print("\nDownload complete.")
else:
    print("ZIP already exists — skipping download.")


--2026-03-27 20:52:20--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.101.207, 142.250.141.207, 142.251.2.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.101.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 108445099632 (101G) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/piano_amt/maestro-v3.0.0.zip’

/content/drive/MyDr 100%[===================>] 101.00G  62.4MB/s    in 32m 21s 

2026-03-27 21:24:42 (53.3 MB/s) - ‘/content/drive/MyDrive/piano_amt/maestro-v3.0.0.zip’ saved [108445099632/108445099632]


Download complete.


## Unzip

Extracts the ZIP to `DRIVE_ROOT/maestro-v3.0.0/`.  
**Expected time: 5–15 minutes**.

In [ ]:

if not os.path.exists(MAESTRO_DIR):
    print("Extracting ZIP...")
    !unzip -q "{ZIP_PATH}" -d "{DRIVE_ROOT}"
    print("Unzipped ✓")
else:
    print("Already unzipped ✓")


Extracting ZIP...
Unzipped ✓


## Verify dataset

In [ ]:

import pandas as pd
import glob

csv_files = sorted(glob.glob(f"{MAESTRO_DIR}/*.csv"))
assert csv_files, f"No CSV found in {MAESTRO_DIR}"

df = pd.read_csv(csv_files[0])
print(f"CSV: {csv_files[0]}")
print(f"Columns: {list(df.columns)}")
print()

for split in ["train", "validation", "test"]:
    n = len(df[df["split"] == split])
    print(f"  {split:12s}: {n:4d} files")

total_hrs = df["duration"].sum() / 3600
print(f"\nTotal duration: {total_hrs:.1f} hours")
print(f"Total files   : {len(df)}")


CSV: /content/drive/MyDrive/piano_amt/maestro-v3.0.0/maestro-v3.0.0.csv
Columns: ['canonical_composer', 'canonical_title', 'split', 'year', 'midi_filename', 'audio_filename', 'duration']

  train       :  962 files
  validation  :  137 files
  test        :  177 files

Total duration: 198.7 hours
Total files   : 1276


## Optional: use a small subset for testing

When calling `MAESTRODataset` or `get_dataloader`, pass `max_files=N` to limit
the dataset to the first N files.  This is useful for quick testing before
building the full cache:

```python
ds = MAESTRODataset(MAESTRO_DIR, 'train', cache_dir, max_files=5)
```

The full dataset has 967 training files (~178 hours).  
Start with `max_files=3` to verify the pipeline, then build the full cache.

## Estimate cache size

In [ ]:
import glob
import pandas as pd



csv_files = sorted(glob.glob(f"{MAESTRO_ROOT}/*.csv"))
df = pd.read_csv(csv_files[0])

n_train = len(df[df["split"] == "train"])
n_val   = len(df[df["split"] == "validation"])
n_test  = len(df[df["split"] == "test"])
n_total = n_train + n_val + n_test

total_hrs = df["duration"].sum() / 3600

print(f"Files: train={n_train}, val={n_val}, test={n_test}, total={n_total}")
print(f"Total audio duration: {total_hrs:.1f} hours")
print(f"Estimated cache size: ~{total_hrs * 120:.0f} MB  (≈120 MB/hr for NPZ)")
print()
print("Note: Actual size depends on piece length. 15 GB is a safe upper bound.")

Files: train=962, val=137, test=177, total=1276
Total audio duration: 198.7 hours
Estimated cache size: ~23838 MB  (≈120 MB/hr for NPZ)

Note: Actual size depends on piece length. 15 GB is a safe upper bound.


## Quick test on 3 files first

Always verify on a small subset before running the full cache build.

In [ ]:
import numpy as np
from pathlib import Path
from src.dataset import preprocess_and_cache, load_from_cache, _cache_path
from src.constants import (
    MAESTRO_AUDIO_COL, MAESTRO_MIDI_COL, MAESTRO_SPLIT_COL, N_MELS
)

train_rows = df[df[MAESTRO_SPLIT_COL] == "train"].head(3)

for _, row in train_rows.iterrows():
    audio_path = str(Path(MAESTRO_ROOT) / row[MAESTRO_AUDIO_COL])
    midi_path  = str(Path(MAESTRO_ROOT) / row[MAESTRO_MIDI_COL])
    cp         = _cache_path(audio_path, CACHE_DIR)

    print(f"\nProcessing: {Path(audio_path).name}")
    if not cp.exists():
        preprocess_and_cache(audio_path, midi_path, cp)
        print(f"  Cached → {cp}")
    else:
        print(f"  Already cached: {cp}")

    # Verify the cache loads correctly
    data = load_from_cache(cp)
    mel = data['mel']  # (229, T)
    assert mel.shape[0] == N_MELS, f"mel.shape[0]={mel.shape[0]}, expected {N_MELS}"
    assert data['onset'].shape[1] == 88
    print(f"  mel.shape    = {tuple(mel.shape)}")
    print(f"  onset.shape  = {tuple(data['onset'].shape)}")
    print(f"  mel range    = [{mel.min():.2f}, {mel.max():.2f}]")

print("\n✓ Quick test passed — safe to build full cache.")


Processing: MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.wav
  Cached → /content/drive/MyDrive/piano_amt/cache/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.npz
  mel.shape    = (229, 22009)
  onset.shape  = (22009, 88)
  mel range    = [-14.51, 9.62]

Processing: MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MID--AUDIO_03_R2_2008_wav--2.wav
  Cached → /content/drive/MyDrive/piano_amt/cache/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MID--AUDIO_03_R2_2008_wav--2.npz
  mel.shape    = (229, 23901)
  onset.shape  = (23901, 88)
  mel range    = [-13.89, 10.12]

Processing: MIDI-Unprocessed_066_PIANO066_MID--AUDIO-split_07-07-17_Piano-e_3-02_wav--3.wav
  Cached → /content/drive/MyDrive/piano_amt/cache/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-split_07-07-17_Piano-e_3-02_wav--3.npz
  mel.shape    = (229, 14570)
  onset.shape  = (14570, 88)
  mel range    = [-15.37, 10.11]

✓ Quick test passed — safe to build full cache.


## Build full cache

This cell processes all train / validation / test files.  
**Expected time: ~25–40 minutes on T4 GPU.**  
Already-cached files are automatically skipped, so it is safe to interrupt
and resume.

In [ ]:
from src.dataset import build_cache

build_cache(
    maestro_root=MAESTRO_ROOT,
    cache_dir=CACHE_DIR,
    splits=("train", "validation", "test"),
)

print("\n✓ Full cache build complete!")

[train]: 962 files


Cache train: 100%|██████████| 962/962 [1:34:48<00:00,  5.91s/it]


  Split 'train' cached successfully.
[validation]: 137 files


Cache validation: 100%|██████████| 137/137 [11:08<00:00,  4.88s/it]


  Split 'validation' cached successfully.
[test]: 177 files


Cache test: 100%|██████████| 177/177 [11:37<00:00,  3.94s/it]

  Split 'test' cached successfully.

✓ Full cache build complete!


## Verify cache

In [ ]:
import glob
import numpy as np

npz_files = sorted(glob.glob(f"{CACHE_DIR}/*.npz"))
print(f"Total .npz files in cache: {len(npz_files)}")

# Load and inspect the first file
if npz_files:
    sample_file = npz_files[0]
    print(f"\nInspecting: {sample_file}")
    data = np.load(sample_file)
    for k in data.files:
        v = data[k]
        print(f"  {k:12s}: shape={v.shape}  dtype={v.dtype}")

    mel = data["mel"]
    assert mel.shape[0] == 229, f"mel.shape[0]={mel.shape[0]}, expected 229"
    assert data["onset"].shape[1] == 88
    print(f"\n✓ Cache structure verified.")
else:
    print("WARNING: No .npz files found in cache!")

Total .npz files in cache: 1276

Inspecting: /content/drive/MyDrive/piano_amt/cache/MIDI-UNPROCESSED_01-03_R1_2014_MID--AUDIO_01_R1_2014_wav--1.npz
  mel         : shape=(229, 4303)  dtype=float32
  onset       : shape=(4303, 88)  dtype=float32
  frame       : shape=(4303, 88)  dtype=float32
  offset      : shape=(4303, 88)  dtype=float32
  velocity    : shape=(4303, 88)  dtype=float32
  sr          : shape=()  dtype=int32

✓ Cache structure verified.


## Cache is complete

The `.npz` files are stored on Google Drive at:
```
/content/drive/MyDrive/piano_amt/cache/
```

Every future Colab session starts from `03_verify_pipeline.ipynb` — which
mounts Drive and loads directly from the cache without re-running this notebook.

If you ever need to regenerate the cache (e.g. after changing mel parameters),
delete the `cache/` folder on Drive and re-run this notebook.

In [20]:
from pathlib import Path

from src.constants import *
from src.audio import load_audio, wav_to_log_mel, load_audio_as_log_mel
from src.midi import midi_path_to_rolls
from src.dataset import MAESTRODataset, load_from_cache
from src.dataloader import get_dataloader

## Check 1: Constants

Asserts that all critical hyperparameters have their expected values from
Hawthorne 2018a §3 and jongwook/onsets-and-frames.

In [ ]:
assert N_MELS == 229,             f"N_MELS={N_MELS}"
assert SAMPLE_RATE == 16000,      f"SAMPLE_RATE={SAMPLE_RATE}"
assert HOP_LENGTH == 512,         f"HOP_LENGTH={HOP_LENGTH}"
assert abs(FRAMES_PER_SECOND - 31.25) < 1e-6
assert MAX_SEGMENT_FRAMES == 640, f"MAX_SEGMENT_FRAMES={MAX_SEGMENT_FRAMES}"
assert N_KEYS == 88,              f"N_KEYS={N_KEYS}"
assert MIN_MIDI == 21,            f"MIN_MIDI={MIN_MIDI}"
assert MAX_MIDI == 108,           f"MAX_MIDI={MAX_MIDI}"

print(f"N_MELS         = {N_MELS}")
print(f"SAMPLE_RATE    = {SAMPLE_RATE}")
print(f"HOP_LENGTH     = {HOP_LENGTH}")
print(f"FRAMES_PER_SEC = {FRAMES_PER_SECOND}")
print(f"MAX_SEG_FRAMES = {MAX_SEGMENT_FRAMES}")
print(f"N_KEYS         = {N_KEYS}")
print(f"MIDI range     = [{MIN_MIDI}, {MAX_MIDI}]")
print("\n✓ Check 1: Constants OK")

N_MELS         = 229
SAMPLE_RATE    = 16000
HOP_LENGTH     = 512
FRAMES_PER_SEC = 31.25
MAX_SEG_FRAMES = 640
N_KEYS         = 88
MIDI range     = [21, 108]

✓ Check 1: Constants OK


## Check 2: Load one mel from cache

In [ ]:
npz_files = sorted(glob.glob(f"{CACHE_DIR}/*.npz"))
assert npz_files, f"No .npz files found in {CACHE_DIR}. Run 02_build_cache.ipynb first."

data = load_from_cache(npz_files[0])

mel = data['mel']  # (229, T)
print(f"File       : {Path(npz_files[0]).name}")
print(f"mel.shape  : {tuple(mel.shape)}")
print(f"mel.dtype  : {mel.dtype}")
print(f"mel.min()  : {mel.min().item():.4f}")
print(f"mel.max()  : {mel.max().item():.4f}")
print(f"mel.mean() : {mel.mean().item():.4f}")

assert mel.shape[0] == N_MELS,  f"mel.shape[0]={mel.shape[0]}"
assert mel.shape[1] > 0,        "mel has zero frames"
assert data['onset'].shape[1] == N_KEYS
assert data['frame'].shape == data['onset'].shape

print("\n✓ Check 2: Cache loading OK")

File       : MIDI-UNPROCESSED_01-03_R1_2014_MID--AUDIO_01_R1_2014_wav--1.npz
mel.shape  : (229, 4303)
mel.dtype  : torch.float32
mel.min()  : -14.3398
mel.max()  : 9.1315
mel.mean() : -4.8882

✓ Check 2: Cache loading OK


## Check 3: MAESTRODataset __getitem__

In [ ]:
ds = MAESTRODataset(
    maestro_root=MAESTRO_ROOT,
    split="train",
    cache_dir=CACHE_DIR,
    segment=True,
    max_files=5,
)
print(f"Dataset size: {len(ds)} files")

item = ds[0]
print("\nItem contents:")
for k, v in item.items():
    if hasattr(v, 'shape'):
        print(f"  {k:12s}: {tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:12s}: {v}")

assert item['mel'].shape    == (N_MELS, MAX_SEGMENT_FRAMES)
assert item['onset'].shape  == (MAX_SEGMENT_FRAMES, N_KEYS)
assert item['frame'].shape  == (MAX_SEGMENT_FRAMES, N_KEYS)
assert item['offset'].shape == (MAX_SEGMENT_FRAMES, N_KEYS)
assert item['velocity'].shape == (MAX_SEGMENT_FRAMES, N_KEYS)
assert isinstance(item['audio_path'], str)

print("\n✓ Check 3: Dataset __getitem__ OK")

Dataset size: 5 files

Item contents:
  mel         : (229, 640)  dtype=torch.float32
  onset       : (640, 88)  dtype=torch.float32
  frame       : (640, 88)  dtype=torch.float32
  offset      : (640, 88)  dtype=torch.float32
  velocity    : (640, 88)  dtype=torch.float32
  audio_path  : /content/drive/MyDrive/piano_amt/maestro-v3.0.0/2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.wav

✓ Check 3: Dataset __getitem__ OK


## Check 4: DataLoader batch

In [ ]:
loader = get_dataloader(
    maestro_root=MAESTRO_ROOT,
    split="train",
    batch_size=2,
    num_workers=0,
    cache_dir=CACHE_DIR,
    max_files=5,
    use_augmentation=False,
    pin_memory=False,
)

batch = next(iter(loader))
B = batch['mel'].shape[0]
print(f"Batch size: {B}")
print("\nBatch contents:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:12s}: {tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:12s}: {v}")

assert batch['mel'].shape    == (B, N_MELS, MAX_SEGMENT_FRAMES)
assert batch['onset'].shape  == (B, MAX_SEGMENT_FRAMES, N_KEYS)
assert batch['frame'].shape  == (B, MAX_SEGMENT_FRAMES, N_KEYS)
assert batch['offset'].shape == (B, MAX_SEGMENT_FRAMES, N_KEYS)

print("\n✓ Check 4: DataLoader batch shapes OK")

Batch size: 2

Batch contents:
  mel         : (2, 229, 640)  dtype=torch.float32
  onset       : (2, 640, 88)  dtype=torch.float32
  frame       : (2, 640, 88)  dtype=torch.float32
  offset      : (2, 640, 88)  dtype=torch.float32
  velocity    : (2, 640, 88)  dtype=torch.float32
  audio_path  : ['/content/drive/MyDrive/piano_amt/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MID--AUDIO_21_R1_2004_01_Track01_wav.wav', '/content/drive/MyDrive/piano_amt/maestro-v3.0.0/2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MID--AUDIO_03_R2_2008_wav--2.wav']

✓ Check 4: DataLoader batch shapes OK


## Check 5: Run full verify script

In [ ]:
!python /content/AMT_FYP/piano_amt/scripts/verify_pipeline.py \
    --maestro_root {MAESTRO_ROOT} \
    --max_files 3

Piano AMT Pipeline Verification
MAESTRO root: /content/drive/MyDrive/piano_amt/maestro-v3.0.0
Test file  : MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.wav
[1] check_constants
  ✓ OK
[2] check_audio
  ✓ OK
[3] check_midi
  ✓ OK
[4] check_dataset
  ✓ OK
[5] check_dataloader
  ✓ OK

ALL CHECKS PASSED ✓
